# Mistral-Small-3.1-24B-GoEmotions — Inference from Hugging Face

This notebook downloads the model from `TextMiningStories/Mistral-Small-3.1-24B-goemotions` on Hugging Face and runs inference. No local model directory is required.

Run cells top to bottom. Cell 3 loads the model (~1-2 min on GPU).

In [4]:
# Cell 1 — Install dependencies (skip if already installed)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

0

In [5]:
# Cell 2 — Configuration
import sys, os, importlib
import pandas as pd
import numpy as np

# ── Edit if needed ────────────────────────────────────────────────────────
HF_REPO_ID  = "TextMiningStories/Mistral-Small-3.1-24B-goemotions"
HF_TOKEN    = ""     # leave empty — repo is public

# Download destination — files are saved directly here (no symlinks, works on Windows)
LOCAL_DIR   = r"C:\Users\587978\Research\hf_models\Mistral-Small-3.1-24B-goemotions"
# ──────────────────────────────────────────────────────────────────────────

# Labels with < 100 test samples — F1 scores inflated, not reliable for production
UNRELIABLE_LABELS = {"relief", "embarrassment", "nervousness", "pride", "remorse", "grief"}

print(f"Repo      : https://huggingface.co/{HF_REPO_ID}")
print(f"Local dir : {LOCAL_DIR}")
print(f"Unreliable: {sorted(UNRELIABLE_LABELS)}")

Repo      : https://huggingface.co/TextMiningStories/Mistral-Small-3.1-24B-goemotions
Local dir : C:\Users\587978\Research\hf_models\Mistral-Small-3.1-24B-goemotions
Unreliable: ['embarrassment', 'grief', 'nervousness', 'pride', 'relief', 'remorse']


In [6]:
# Cell 3 — Download model from Hugging Face
# Files are saved directly into LOCAL_DIR (no symlinks — avoids Windows privilege error).
# Subsequent runs skip files that are already present and up to date.
from huggingface_hub import snapshot_download

os.makedirs(LOCAL_DIR, exist_ok=True)

hf_dir = snapshot_download(
    repo_id=HF_REPO_ID,
    token=HF_TOKEN or None,
    local_dir=LOCAL_DIR,          # download directly — no symlinks needed
)

print(f"Downloaded to : {hf_dir}")
print(f"Contents      : {os.listdir(hf_dir)}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

focal_config.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Downloaded to : C:\Users\587978\Research\hf_models\Mistral-Small-3.1-24B-goemotions
Contents      : ['.cache', '.gitattributes', 'config.json', 'focal_config.json', 'head_weights.pt', 'infer.py', 'lora_adapter', 'README.md']


In [7]:
# Cell 4 — Load EmotionClassifier from the downloaded directory (~1-2 min)
if hf_dir not in sys.path:
    sys.path.insert(0, hf_dir)

import infer as infer_module
importlib.reload(infer_module)   # ensure we use the HF copy, not any cached local import

clf = infer_module.EmotionClassifier(hf_dir)

print(f"Device    : {clf.device}")
print(f"Labels ({len(clf.labels)}): {clf.labels}")
print(f"Threshold : {clf.threshold}")
print("\nModel ready.")

c:\Users\587978\.conda\envs\epu\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0602 13:55:14.138000 22860 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!



[tensorflow|WARNING]From c:\Users\587978\.conda\envs\epu\Lib\site-packages\tensorflow_hub\resolver.py:120: The name tf.gfile.MakeDirs is deprecated. Please use tf.io.gfile.makedirs instead.



[tensorflow|WARNING]From c:\Users\587978\.conda\envs\epu\Lib\site-packages\tensorflow_hub\module_v2.py:126: The name tf.saved_model.load_v2 is deprecated. Please use tf.compat.v2.saved_model.load instead.



==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX 6000 Ada Generation. Num GPUs = 1. Max memory: 47.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device    : cuda
Labels (28): ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']
Threshold : 0.5

Model ready.


In [8]:
# Cell 5 — Single-text prediction
text = "I just got promoted at work — I can't believe it, this is amazing!"

result = clf.predict([text])[0]

print(f"Input  : {text}")
print()
print("Detected emotions (sorted by confidence):")
for emotion, score in sorted(result.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 20)
    print(f"  {emotion:<20} {score:.4f}  {bar}")

Input  : I just got promoted at work — I can't believe it, this is amazing!

Detected emotions (sorted by confidence):
  admiration           0.9898  ███████████████████


In [9]:
# Cell 6 — Batch prediction on diverse emotion examples
TEST_TEXTS = [
    "I can't stop laughing, that was the funniest thing I've ever seen!",       # amusement
    "How dare you lie to me like that — I am absolutely furious!",              # anger
    "I'm so worried about the exam tomorrow, my hands are shaking.",            # nervousness / fear
    "Thank you so much, you really saved me today.",                            # gratitude
    "I don't understand why this keeps happening, it makes no sense.",          # confusion
    "I just want to curl up and cry, everything feels hopeless.",               # sadness
    "This sunset is absolutely breathtaking, I feel so lucky to be alive.",     # joy / admiration
    "I'm really disappointed — I expected so much more from this.",             # disappointment
    "That behaviour is completely unacceptable and morally wrong.",             # disapproval / disgust
    "I have a good feeling about this, things are going to work out.",          # optimism
]

predictions = clf.predict(TEST_TEXTS)

rows = []
for text, pred in zip(TEST_TEXTS, predictions):
    top = sorted(pred.items(), key=lambda x: x[1], reverse=True)
    rows.append({
        "text":         text[:60] + "...",
        "top_emotion":  top[0][0] if top else "",
        "top_score":    round(top[0][1], 4) if top else 0,
        "all_detected": ", ".join(f"{e}({s:.2f})" for e, s in top),
    })

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,text,top_emotion,top_score,all_detected
0,"I can't stop laughing, that was the funniest thing I've ever...",amusement,0.9975,amusement(1.00)
1,How dare you lie to me like that — I am absolutely furious!...,anger,0.9272,anger(0.93)
2,"I'm so worried about the exam tomorrow, my hands are shaking...",sadness,0.6145,sadness(0.61)
3,"Thank you so much, you really saved me today....",gratitude,1.0000,gratitude(1.00)
4,"I don't understand why this keeps happening, it makes no sen...",confusion,0.9999,confusion(1.00)
5,"I just want to curl up and cry, everything feels hopeless....",sadness,0.9685,sadness(0.97)
6,"This sunset is absolutely breathtaking, I feel so lucky to b...",admiration,1.0000,admiration(1.00)
7,I'm really disappointed — I expected so much more from this....,disappointment,0.8452,disappointment(0.85)
8,That behaviour is completely unacceptable and morally wrong....,,0.0000,
9,"I have a good feeling about this, things are going to work o...",,0.0000,


In [10]:
# Cell 7 — Full probability scores for one text (all 28 labels)
PROBE_TEXT = "I just want to curl up and cry, everything feels hopeless."

all_scores = clf.predict([PROBE_TEXT], threshold=0.0)[0]

df_all = pd.DataFrame(
    sorted(all_scores.items(), key=lambda x: x[1], reverse=True),
    columns=["emotion", "probability"]
)
df_all["above_threshold"] = df_all["probability"] >= clf.threshold
df_all["unreliable"]      = df_all["emotion"].isin(UNRELIABLE_LABELS)
df_all["bar"]             = df_all["probability"].apply(lambda p: "█" * int(p * 30))

print(f"Text: {PROBE_TEXT}\n")
df_all

Text: I just want to curl up and cry, everything feels hopeless.



,emotion,probability,above_threshold,unreliable,bar
0,sadness,0.9695,True,False,█████████████████████████████


In [11]:
# Cell 8 — Threshold comparison (0.30 / 0.50 / 0.70)
THRESHOLD_TEXT = "This is so unfair, I'm really angry and upset about what happened."

thresholds = [0.30, 0.50, 0.70]
results = {thr: clf.predict([THRESHOLD_TEXT], threshold=thr)[0] for thr in thresholds}

all_emotions = sorted(set().union(*[r.keys() for r in results.values()]))
rows = []
for emotion in all_emotions:
    row = {"emotion": emotion}
    for thr in thresholds:
        row[f"thr={thr}"] = results[thr].get(emotion, "—")
    rows.append(row)

print(f"Text: {THRESHOLD_TEXT}\n")
pd.DataFrame(rows).set_index("emotion")

Text: This is so unfair, I'm really angry and upset about what happened.



,thr=0.3,thr=0.5,thr=0.7
emotion,,,
anger,0.9863,0.9863,0.9863


In [12]:
# Cell 9 — Unreliable-label filtering
FILTER_TEXT = "I'm so ashamed of what I did — I wish I could take it back."

raw      = clf.predict([FILTER_TEXT])[0]
filtered = {k: v for k, v in raw.items() if k not in UNRELIABLE_LABELS}

print(f"Text: {FILTER_TEXT}\n")
print("--- Raw output (includes unreliable labels) ---")
for e, s in sorted(raw.items(), key=lambda x: x[1], reverse=True):
    flag = "  <-- UNRELIABLE" if e in UNRELIABLE_LABELS else ""
    print(f"  {e:<20} {s:.4f}{flag}")

print()
print("--- Filtered output (safe for production) ---")
if filtered:
    for e, s in sorted(filtered.items(), key=lambda x: x[1], reverse=True):
        print(f"  {e:<20} {s:.4f}")
else:
    print("  (no reliable emotions detected above threshold)")

Text: I'm so ashamed of what I did — I wish I could take it back.

--- Raw output (includes unreliable labels) ---
  disappointment       0.7976

--- Filtered output (safe for production) ---
  disappointment       0.7976


In [13]:
# Cell 10 — Your own texts
# ── Edit the list below and re-run this cell ──────────────────────────────
MY_TEXTS = [
    "Write your first sentence here.",
    "Write your second sentence here.",
    "Write your third sentence here.",
]
MY_THRESHOLD      = 0.50   # adjust as needed
FILTER_UNRELIABLE = True   # set False to include unreliable labels
# ──────────────────────────────────────────────────────────────────────────

my_preds = clf.predict(MY_TEXTS, threshold=MY_THRESHOLD)

for text, pred in zip(MY_TEXTS, my_preds):
    if FILTER_UNRELIABLE:
        pred = {k: v for k, v in pred.items() if k not in UNRELIABLE_LABELS}
    print(f"Text   : {text}")
    if pred:
        for e, s in sorted(pred.items(), key=lambda x: x[1], reverse=True):
            print(f"         {e:<20} {s:.4f}")
    else:
        print("         (no emotions detected above threshold)")
    print()

Text   : Write your first sentence here.
         (no emotions detected above threshold)

Text   : Write your second sentence here.
         neutral              0.9985

Text   : Write your third sentence here.
         (no emotions detected above threshold)

